In [1]:
# ============ EfficientNet-B0 (timm: efficientnet_b0) | 3-fold CV | test on Common_test_20 ============
# Transfer learning: backbone frozen, classifier + last block(s) fine-tuned.
#   - BatchNorm of FROZEN layers stays in eval mode
#   - Optimizer: classifier at lr, unfrozen backbone layers at lr x 0.1
import os, sys, re, json, time, random, copy, threading, tempfile, shutil, uuid
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import psutil
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, roc_curve, auc, classification_report)
from sklearn.preprocessing import label_binarize

# ---------------- Google Drive-safe torch.save (fixes "Operation not supported") ----------------
if not getattr(torch.save, "_drive_safe", False):
    _orig_save = torch.save

    def _drive_safe_save(obj, f, *args, **kwargs):
        if not isinstance(f, (str, os.PathLike)) or not str(f).startswith("/content/drive"):
            return _orig_save(obj, f, *args, **kwargs)
        f = str(f)
        tmp_local = f"/content/_ckpt_tmp_{uuid.uuid4().hex}.pt"
        _orig_save(obj, tmp_local, *args, **kwargs)                      # 1) write on fast local disk
        mirror = "/content/local_ckpt_mirror/" + f.replace("/content/drive/MyDrive/", "")
        os.makedirs(os.path.dirname(mirror), exist_ok=True)
        shutil.copyfile(tmp_local, mirror)                               # 2) local backup copy
        ok = False
        for attempt in range(5):                                         # 3) copy to Drive with retries
            try:
                shutil.copyfile(tmp_local, f + ".tmp"); os.replace(f + ".tmp", f); ok = True; break
            except Exception as e:
                print(f"  [safe_save] Drive write failed (attempt {attempt + 1}/5): {e}"); time.sleep(3)
        os.remove(tmp_local)
        if not ok:
            print(f"  [safe_save] WARNING: could not write to Drive: {f}\n  local copy kept at: {mirror}")

    _drive_safe_save._drive_safe = True
    torch.save = _drive_safe_save

# ---------------- QUICK SETTINGS ----------------
MODEL_NAME     = "efficientnet_b0"         # timm model name
MODEL_SOURCE   = "timm"
RESUME_RUN_DIR = None                # after a disconnect: paste the existing run folder path here and re-run
VERIFY_ONLY    = False               # True = only check the CSVs, no training

CFG = {
    "data_root":   "/content/drive/MyDrive/Dataset Files/Turmeric Dataset/Original DataSet",
    "folds_dir":   "/content/drive/MyDrive/Dataset Files/Turmeric Dataset/3Fold_CV",
    "test_dir":    "/content/drive/MyDrive/Dataset Files/Turmeric Dataset/Common_test_20",
    "output_root": "/content/drive/MyDrive/Colab Notebooks/Thesis notebook",
    "resume_run_dir": RESUME_RUN_DIR,
    "model_name": MODEL_NAME, "model_source": MODEL_SOURCE,
    "pretrained": True, "dropout": 0.3, "num_classes_expected": 4,
    "freeze_backbone": True,
    "unfreeze_last_blocks": 1,        # 0 = classifier only | 1 = last stage + head layers + classifier | 2.. = more
    "model_overrides": {                          # applied automatically for the selected model
        "efficientnet_b0": {"img_size": 224},
    },
    "img_size": 224, "batch_size": 32, "max_epochs": 30, "lr": 1e-4, "weight_decay": 1e-4,
    "optimizer": "AdamW", "seed": 42,
    "scheduler": {"type": "ReduceLROnPlateau", "mode": "min", "factor": 0.5, "patience": 3, "min_lr": 1e-7},
    "early_stopping": {"monitor": "val_loss", "patience": 7, "min_delta": 1e-4},
    "use_amp": True, "use_class_weights": False, "num_workers": 2, "cache_images_in_ram": True,
    "imagenet_mean": [0.485, 0.456, 0.406], "imagenet_std": [0.229, 0.224, 0.225],
    "augmentation": {"rotation_factor": 0.10, "horizontal_flip": True, "vertical_flip": True,
                     "zoom_factor": 0.10, "shear_x_factor": 0.10, "shear_y_factor": 0.10},
    "bench": {"warmup": 20, "runs": 100, "energy_seconds": 30, "assumed_cpu_tdp_watts": 15.0},
}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
LOG_FILE = None


def log(*a):
    msg = " ".join(str(x) for x in a)
    print(msg, flush=True)
    if LOG_FILE:
        with open(LOG_FILE, "a") as f:
            f.write(msg + "\n")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def jdump(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))


# ---------------- MANIFEST LOADING + VERIFICATION ----------------
PATH_CANDS = ["image_path", "filepath", "file_path", "path", "image", "filename", "file", "img_path", "image_name"]
LABEL_CANDS = ["label", "class", "class_name", "target", "category", "y", "label_name"]
_INDEX = {}          # root folder -> {filename: [full paths]}


def build_index(root):
    if root not in _INDEX:
        idx = {}
        for r, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs if d != "CSV_Manifests"]
            for fn in files:
                if Path(fn).suffix.lower() in IMG_EXT:
                    idx.setdefault(fn, []).append(os.path.join(r, fn))
        _INDEX[root] = idx
    return _INDEX[root]


def resolve_path(p, roots=None):
    roots = roots or [CFG["data_root"]]
    p = str(p).strip().replace("\\", "/")
    if os.path.isabs(p) and os.path.exists(p):
        return p
    for root in roots:
        c = os.path.join(root, p)
        if os.path.exists(c):
            return c
        cand = build_index(root).get(os.path.basename(p), [])
        if len(cand) == 1:
            return cand[0]
        if len(cand) > 1:
            parent = os.path.basename(os.path.dirname(p))
            m = [x for x in cand if os.path.basename(os.path.dirname(x)) == parent]
            if len(m) == 1:
                return m[0]
    return None


def read_manifest(csv_path, roots=None):
    df = pd.read_csv(csv_path)
    cols = {c.lower().strip(): c for c in df.columns}
    pc = next((cols[c] for c in PATH_CANDS if c in cols), None)
    lc = next((cols[c] for c in LABEL_CANDS if c in cols), None)
    if pc is None:
        raise ValueError(f"[{csv_path}] cannot find an image-path column. Columns: {list(df.columns)}")
    out = pd.DataFrame({"orig_path": df[pc].astype(str)})
    out["path"] = out["orig_path"].map(lambda x: resolve_path(x, roots))
    if lc is None:
        log(f"  WARNING [{Path(csv_path).name}] no label column; using parent folder name as label.")
        out["label_raw"] = out["orig_path"].map(lambda x: os.path.basename(os.path.dirname(x.replace("\\", "/"))))
    else:
        out["label_raw"] = df[lc]
    missing = out[out["path"].isna()]
    if len(missing):
        raise FileNotFoundError(f"[{csv_path}] {len(missing)} image paths could not be resolved. First 10:\n"
                                + "\n".join(missing["orig_path"].head(10)))
    return out


def read_test_manifest(test_path):
    """Test set can be: a CSV file, a folder containing a CSV, or a folder with class sub-folders of images."""
    tp = Path(test_path)
    if tp.is_file():
        return read_manifest(str(tp), roots=[str(tp.parent), CFG["data_root"]])
    if not tp.is_dir():
        raise FileNotFoundError(f"Test path not found: {test_path}")
    csvs = sorted(tp.rglob("*.csv"))
    if csvs:
        log(f"  Test set: using manifest {csvs[0]}")
        return read_manifest(str(csvs[0]), roots=[str(tp), CFG["data_root"]])
    rows = []
    for root, _, files in os.walk(tp):
        for fn in sorted(files):
            if Path(fn).suffix.lower() in IMG_EXT:
                full = os.path.join(root, fn)
                rows.append((full, full, os.path.basename(root)))
    if not rows:
        raise FileNotFoundError(f"No CSV and no images found in test folder: {test_path}")
    df = pd.DataFrame(rows, columns=["orig_path", "path", "label_raw"])
    if any(os.path.normpath(os.path.dirname(p)) == os.path.normpath(str(tp)) for p in df["path"]):
        raise ValueError("Test folder has images directly inside it. Put them in class sub-folders "
                         "(one per class) or provide a CSV with path + label columns.")
    log(f"  Test set: built from class sub-folders in {tp} ({len(df)} images)")
    return df


def find_fold_files(folder):
    folds = {}
    for f in sorted(Path(folder).rglob("*.csv")):
        n = f.name.lower()
        m = re.search(r"fold[_\-\s]?(\d+)", n) or re.search(r"fold[_\-\s]?(\d+)", str(f.parent).lower())
        if not m:
            continue
        split = "train" if "train" in n else ("val" if ("val" in n) else None)
        if split:
            folds.setdefault(int(m.group(1)), {})[split] = f
    ok = {k: v for k, v in folds.items() if "train" in v and "val" in v}
    if not ok:
        raise FileNotFoundError(f"Could not pair train/val CSVs in {folder}. Files found: "
                                f"{[str(p.name) for p in Path(folder).rglob('*.csv')]}")
    return dict(sorted(ok.items()))


def build_label_map(dfs):
    all_df = pd.concat(dfs)
    raw = all_df["label_raw"]
    if pd.api.types.is_numeric_dtype(raw):
        uniq = sorted(raw.unique())
        names = [all_df.loc[raw == u, "path"].map(lambda x: os.path.basename(os.path.dirname(x))).mode().iloc[0]
                 for u in uniq]
        if len(set(names)) != len(names):
            names = [str(u) for u in uniq]
    else:
        uniq = sorted(raw.astype(str).str.strip().unique())
        names = list(uniq)
    return {u: i for i, u in enumerate(uniq)}, names


def apply_labels(df, mapping, names=None):
    raw = df["label_raw"]
    df = df.copy()
    if pd.api.types.is_numeric_dtype(raw):
        y = raw.map(mapping)
    else:
        s = raw.astype(str).str.strip()
        y = s.map(mapping)
        if y.isna().any() and names is not None:          # e.g. numeric labels in folds, folder names in test
            y = s.map({n: i for i, n in enumerate(names)})
    if y.isna().any():
        bad = sorted(set(raw[y.isna()].astype(str)))[:10]
        raise ValueError(f"Some labels not found in the label map: {bad}")
    df["y"] = y.astype(int)
    return df


def load_and_verify(run_dir):
    log("=" * 80); log("STEP 1: verifying dataset + CSV manifests"); log("=" * 80)
    idx = build_index(CFG["data_root"])
    n_disk = sum(len(v) for v in idx.values())
    class_dirs = sorted({os.path.basename(os.path.dirname(p)) for v in idx.values() for p in v})
    log(f"Images on disk under data_root: {n_disk}  | folders containing images: {class_dirs}")
    dup_names = {k: len(v) for k, v in idx.items() if len(v) > 1}
    if dup_names:
        log(f"  NOTE: {len(dup_names)} file names appear in more than one folder (resolved by parent folder).")
    fold_files = find_fold_files(CFG["folds_dir"])
    log(f"Folds found: {list(fold_files.keys())}")
    raw_folds = {}
    for k, d in fold_files.items():
        log(f"  fold {k}: train={d['train'].name}  val={d['val'].name}")
        raw_folds[k] = {"train": read_manifest(d["train"]), "val": read_manifest(d["val"])}
    test_raw = read_test_manifest(CFG["test_dir"])
    mapping, class_names = build_label_map([x for k in raw_folds for x in raw_folds[k].values()])
    folds = {k: {s: apply_labels(df, mapping, class_names) for s, df in v.items()} for k, v in raw_folds.items()}
    test_df = apply_labels(test_raw, mapping, class_names)

    log(f"\nClasses ({len(class_names)}): {class_names}")
    if len(class_names) != CFG["num_classes_expected"]:
        log(f"  WARNING: expected {CFG['num_classes_expected']} classes but found {len(class_names)}.")
    report = {"classes": class_names, "images_on_disk": n_disk, "folds": {}}
    test_paths = set(test_df["path"])
    test_names = {os.path.basename(p) for p in test_paths}
    log(f"\nTEST  n={len(test_df)}  per-class={test_df['y'].value_counts().sort_index().tolist()}")
    for k, d in folds.items():
        tr, va = set(d["train"]["path"]), set(d["val"]["path"])
        tr_names = {os.path.basename(p) for p in tr}; va_names = {os.path.basename(p) for p in va}
        r = {"n_train": len(d["train"]), "n_val": len(d["val"]),
             "train_per_class": d["train"]["y"].value_counts().sort_index().tolist(),
             "val_per_class": d["val"]["y"].value_counts().sort_index().tolist(),
             "train_val_overlap": len(tr & va), "train_test_overlap": len(tr & test_paths),
             "val_test_overlap": len(va & test_paths),
             "train_test_filename_overlap": len(tr_names & test_names),
             "val_test_filename_overlap": len(va_names & test_names),
             "duplicate_paths_in_train": int(d["train"]["path"].duplicated().sum()),
             "duplicate_paths_in_val": int(d["val"]["path"].duplicated().sum())}
        report["folds"][k] = r
        log(f"FOLD {k}: train={r['n_train']} {r['train_per_class']} | val={r['n_val']} {r['val_per_class']}")
        log(f"        overlaps -> train/val={r['train_val_overlap']} train/test={r['train_test_overlap']} "
            f"val/test={r['val_test_overlap']}  dups(train/val)={r['duplicate_paths_in_train']}/{r['duplicate_paths_in_val']}")
        if r["train_val_overlap"] or r["train_test_overlap"] or r["val_test_overlap"]:
            log("        !!! DATA LEAKAGE DETECTED - fix the CSVs before trusting results !!!")
        elif r["train_test_filename_overlap"] or r["val_test_filename_overlap"]:
            log(f"        WARNING: same file NAMES appear in train/val and test "
                f"({r['train_test_filename_overlap']}/{r['val_test_filename_overlap']}) - "
                f"check the test images are not copies of training images.")
    union = set(test_paths)
    for d in folds.values():
        union |= set(d["train"]["path"]) | set(d["val"]["path"])
    log(f"\nUnique images referenced by all CSVs + test set: {len(union)} (you said dataset has 865)")
    report["unique_images_in_csvs"] = len(union)
    cnt = folds[min(folds)]["train"]["y"].value_counts()
    ratio = cnt.max() / cnt.min()
    log(f"Class imbalance ratio (fold-1 train, max/min): {ratio:.2f}"
        + ("  -> consider use_class_weights=True" if ratio > 1.5 else "  -> roughly balanced"))
    report["imbalance_ratio_fold1_train"] = float(ratio)
    jdump(report, os.path.join(run_dir, "csv_verification_report.json"))
    log("Verification report saved.\n")
    return folds, test_df, class_names


# ---------------- DATA ----------------
_IMG_CACHE = {}


def load_img(path):
    if CFG["cache_images_in_ram"] and path in _IMG_CACHE:
        return _IMG_CACHE[path]
    img = Image.open(path).convert("RGB").resize((CFG["img_size"], CFG["img_size"]), Image.BILINEAR)
    if CFG["cache_images_in_ram"]:
        _IMG_CACHE[path] = img
    return img


class ImgDS(Dataset):
    def __init__(self, df, tf):
        self.paths, self.y, self.tf = df["path"].tolist(), df["y"].tolist(), tf
        for p in self.paths:          # also validates every image can be opened
            load_img(p)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.tf(load_img(self.paths[i])), self.y[i], i


def make_transforms():
    a = CFG["augmentation"]
    norm = transforms.Normalize(CFG["imagenet_mean"], CFG["imagenet_std"])
    deg = a["rotation_factor"] * 360.0                         # Keras factor 0.10 -> +-36 degrees
    sx = float(np.degrees(np.arctan(a["shear_x_factor"])))     # shear 0.10 -> ~5.71 degrees
    sy = float(np.degrees(np.arctan(a["shear_y_factor"])))
    z = a["zoom_factor"]
    train_tf = transforms.Compose([                            # augmentation: TRAINING images only
        transforms.RandomHorizontalFlip(0.5 if a["horizontal_flip"] else 0.0),
        transforms.RandomVerticalFlip(0.5 if a["vertical_flip"] else 0.0),
        transforms.RandomAffine(degrees=deg, scale=(1 - z, 1 + z), shear=(-sx, sx, -sy, sy), fill=0),
        transforms.ToTensor(), norm])
    eval_tf = transforms.Compose([transforms.ToTensor(), norm])
    return train_tf, eval_tf


def make_loader(ds, shuffle, seed):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle, num_workers=CFG["num_workers"],
                      pin_memory=torch.cuda.is_available(), generator=g,
                      worker_init_fn=lambda w: (np.random.seed(seed + w), random.seed(seed + w)))


# ---------------- MODEL ----------------
def _last_linear_name(model):
    last = None
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear):
            last = name
    if last is None:
        raise ValueError("No nn.Linear classification layer found in this model.")
    return last


def _set_module(model, name, new):
    parts = name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def _unfreeze(module):
    for q in module.parameters():
        q.requires_grad = True


def build_model(num_classes):
    name, src, p = CFG["model_name"], CFG["model_source"], CFG["dropout"]
    if src == "timm":
        import timm
        m = timm.create_model(name, pretrained=CFG["pretrained"], num_classes=num_classes, drop_rate=p)
        head = m.get_classifier()
    elif src == "torchvision":
        m = models.get_model(name, weights="DEFAULT" if CFG["pretrained"] else None)
        last = _last_linear_name(m)
        mods = dict(m.named_modules())
        new_lin = nn.Linear(mods[last].in_features, num_classes)
        parent = mods[last.rpartition(".")[0]]
        tail = last.rpartition(".")[2]
        if isinstance(parent, nn.Sequential) and tail.isdigit() and int(tail) > 0 \
                and isinstance(parent[int(tail) - 1], nn.Dropout):
            parent[int(tail) - 1].p = p
            head = new_lin
        else:
            head = nn.Sequential(nn.Dropout(p), new_lin)
        _set_module(m, last, head)
    else:
        raise ValueError(f"model_source must be 'torchvision' or 'timm', got {src}")

    # ------------- freeze backbone, unfreeze classifier (+ last block(s)) -------------
    if CFG["freeze_backbone"]:
        for q in m.parameters():                  # 1) freeze everything
            q.requires_grad = False
        _unfreeze(head)                           # 2) unfreeze the classifier
        n_last = CFG.get("unfreeze_last_blocks", 0)
        if n_last > 0 and src == "timm":          # 3) unfreeze the last backbone layer(s)
            if hasattr(m, "blocks"):              # EfficientNet family (B0 / B1 / Lite0): stages in m.blocks
                for blk in list(m.blocks)[-n_last:]:
                    _unfreeze(blk)
                for part in ("conv_head", "bn2"):  # layers right after the blocks
                    if hasattr(m, part):
                        _unfreeze(getattr(m, part))
            elif hasattr(m, "features") and isinstance(m.features, nn.Sequential):   # DenseNet121
                dense = [n for n, _ in m.features.named_children() if n.startswith("denseblock")]
                for n in dense[-n_last:]:
                    _unfreeze(getattr(m.features, n))
                if hasattr(m.features, "norm5"):   # final BN after the last dense block
                    _unfreeze(m.features.norm5)
            else:
                log("  WARNING: unknown backbone layout - only the classifier is trainable.")
        tot = sum(q.numel() for q in m.parameters())
        trn = sum(q.numel() for q in m.parameters() if q.requires_grad)
        log(f"  [freeze] trainable {trn:,}/{tot:,} params")
        log(f"  [freeze] trainable layers: "
            f"{sorted({n.rsplit('.', 1)[0] for n, q in m.named_parameters() if q.requires_grad})}")
    return m


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


# ---------------- METRICS / PLOTS ----------------
def safe_auc(y, probs, K, average):
    try:
        return float(roc_auc_score(y, probs, multi_class="ovr", average=average, labels=list(range(K))))
    except ValueError:
        return float("nan")


def compute_metrics(y_true, probs, class_names):
    y_true = np.asarray(y_true); probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum(1, keepdims=True)
    y_pred = probs.argmax(1); K = len(class_names); labels = list(range(K))
    m = {"n_samples": int(len(y_true)), "accuracy": float(accuracy_score(y_true, y_pred))}
    for avg in ("macro", "weighted"):
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average=avg, zero_division=0)
        m[f"precision_{avg}"], m[f"recall_{avg}"], m[f"f1_{avg}"] = float(p), float(r), float(f)
    m["macro_f1"], m["weighted_f1"] = m["f1_macro"], m["f1_weighted"]
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    m["per_class"] = {class_names[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]),
                                       "support": int(s[i]),
                                       "auc": float(roc_auc_score((y_true == i).astype(int), probs[:, i]))
                                       if 0 < (y_true == i).sum() < len(y_true) else float("nan")}
                      for i in range(K)}
    m["roc_auc_ovr_macro"] = safe_auc(y_true, probs, K, "macro")
    m["roc_auc_ovr_weighted"] = safe_auc(y_true, probs, K, "weighted")
    try:
        m["auc_micro"] = float(roc_auc_score(label_binarize(y_true, classes=labels), probs, average="micro"))
    except ValueError:
        m["auc_micro"] = float("nan")
    m["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=labels).tolist()
    m["classification_report"] = classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                                       zero_division=0)
    return m


def plot_confusion(cm, class_names, path, title):
    cm = np.array(cm); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
    for a, data, t, fmt in ((ax[0], cm, "Counts", "d"), (ax[1], cmn, "Row-normalised", ".2f")):
        im = a.imshow(data, cmap="Blues"); a.set_title(f"{title} - {t}")
        a.set_xticks(range(len(class_names))); a.set_yticks(range(len(class_names)))
        a.set_xticklabels(class_names, rotation=45, ha="right"); a.set_yticklabels(class_names)
        a.set_xlabel("Predicted"); a.set_ylabel("True")
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                a.text(j, i, format(data[i, j], fmt), ha="center", va="center",
                       color="white" if data[i, j] > data.max() / 2 else "black")
        fig.colorbar(im, ax=a, fraction=0.046)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def plot_roc(y_true, probs, class_names, path, title):
    K = len(class_names); Y = label_binarize(y_true, classes=list(range(K)))
    curves = [roc_curve(Y[:, i], probs[:, i])[:2] for i in range(K)]
    fpr_all = np.unique(np.concatenate([c[0] for c in curves]))
    mean_tpr = np.mean([np.interp(fpr_all, f, t) for f, t in curves], axis=0)
    plt.figure(figsize=(7, 6))
    for (f, t), n in zip(curves, class_names):
        plt.plot(f, t, label=f"{n} (AUC={auc(f, t):.3f})")
    plt.plot(fpr_all, mean_tpr, "k--", lw=2, label=f"macro (AUC={auc(fpr_all, mean_tpr):.3f})")
    plt.plot([0, 1], [0, 1], "grey", ls=":")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title(title); plt.legend(loc="lower right")
    plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()
    np.savez(path.replace(".png", "_points.npz"), macro_fpr=fpr_all, macro_tpr=mean_tpr,
             **{f"{class_names[i]}_fpr": curves[i][0] for i in range(K)},
             **{f"{class_names[i]}_tpr": curves[i][1] for i in range(K)})


def plot_curves(h, path, title):
    fig, ax = plt.subplots(1, 4, figsize=(21, 4)); e = h["epoch"]
    ax[0].plot(e, h["train_loss"], label="train"); ax[0].plot(e, h["val_loss"], label="val"); ax[0].set_title("Loss")
    ax[1].plot(e, h["train_acc"], label="train"); ax[1].plot(e, h["val_acc"], label="val"); ax[1].set_title("Accuracy")
    ax[2].plot(e, h["val_roc_auc"], label="val ROC-AUC (macro OvR)"); ax[2].set_title("Validation ROC-AUC")
    ax[3].plot(e, h["lr"]); ax[3].set_yscale("log"); ax[3].set_title("Learning rate")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=.3)
        if a.get_legend_handles_labels()[0]:
            a.legend()
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


# ---------------- ENERGY (CodeCarbon optional) ----------------
class Energy:
    def __init__(self, name):
        self.t = None
        try:
            from codecarbon import EmissionsTracker
            try:
                self.t = EmissionsTracker(project_name=name, measure_power_secs=5, save_to_file=False,
                                          log_level="error", allow_multiple_runs=True)
            except TypeError:
                self.t = EmissionsTracker(project_name=name, measure_power_secs=5, save_to_file=False,
                                          log_level="error")
        except Exception as e:
            log(f"  (codecarbon unavailable: {e}) - energy will be estimated only")

    def start(self):
        if self.t:
            try: self.t.start()
            except Exception: self.t = None

    def stop(self):
        if not self.t:
            return {"energy_kwh": None, "co2_kg": None}
        try:
            co2 = self.t.stop()
            d = getattr(self.t, "final_emissions_data", None)
            kwh = getattr(d, "energy_consumed", None)
            return {"energy_kwh": float(kwh) if kwh is not None else None, "co2_kg": float(co2) if co2 else None}
        except Exception:
            return {"energy_kwh": None, "co2_kg": None}


# ---------------- TRAINING ----------------
def set_train_mode(model, train):
    """Train mode, but keep BatchNorm layers whose weights are FROZEN in eval mode
    so their running statistics are NOT updated. BN layers inside unfrozen blocks keep training."""
    model.train(train)
    if train and CFG["freeze_backbone"]:
        for mod in model.modules():
            if isinstance(mod, nn.modules.batchnorm._BatchNorm) \
                    and not any(p.requires_grad for p in mod.parameters()):
                mod.eval()


def run_epoch(model, loader, criterion, optimizer, scaler, device, train, use_amp, K):
    set_train_mode(model, train)
    tot_loss, correct, n = 0.0, 0, 0
    P, Y = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y, _ in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(x); loss = criterion(out, y)
            if train:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            tot_loss += loss.item() * len(y); n += len(y)
            correct += (out.argmax(1) == y).sum().item()
            P.append(out.float().softmax(1).detach().cpu().numpy()); Y.append(y.cpu().numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    return tot_loss / n, correct / n, safe_auc(Y, P / P.sum(1, keepdims=True), K, "macro")


@torch.no_grad()
def predict(model, ds, device):
    model.eval(); P = []
    for x, _, _ in make_loader(ds, False, CFG["seed"]):
        with torch.autocast(device_type=device.type, enabled=CFG["use_amp"] and device.type == "cuda"):
            P.append(model(x.to(device)).float().softmax(1).cpu().numpy())
    return np.concatenate(P)


def train_fold(k, tr_df, va_df, class_names, run_dir, device):
    fdir = os.path.join(run_dir, f"fold_{k}"); os.makedirs(fdir, exist_ok=True)
    last_p, best_p = os.path.join(fdir, "last.pt"), os.path.join(fdir, "best.pt")
    K = len(class_names); use_amp = CFG["use_amp"] and device.type == "cuda"
    set_seed(CFG["seed"])
    train_tf, eval_tf = make_transforms()
    tr_loader = make_loader(ImgDS(tr_df, train_tf), True, CFG["seed"])
    va_loader = make_loader(ImgDS(va_df, eval_tf), False, CFG["seed"])

    model = build_model(K).to(device)
    total_p, train_p = count_params(model)
    log(f"[fold {k}] model parameters: total {total_p:,} | trainable {train_p:,}")
    weight = None
    if CFG["use_class_weights"]:
        c = np.bincount(tr_df["y"], minlength=K)
        weight = torch.tensor(len(tr_df) / (K * c), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weight)

    # ---- optimizer: classifier at full lr, unfrozen backbone layer(s) at lr x 0.1 ----
    head_ids = {id(q) for q in model.get_classifier().parameters()}
    bb = [q for q in model.parameters() if q.requires_grad and id(q) not in head_ids]
    hd = [q for q in model.parameters() if q.requires_grad and id(q) in head_ids]
    groups = [{"params": hd, "lr": CFG["lr"]}]                   # group 0 = classifier head
    if bb:
        groups.append({"params": bb, "lr": CFG["lr"] * 0.1})     # group 1 = unfrozen backbone layer(s)
    optimizer = torch.optim.AdamW(groups, weight_decay=CFG["weight_decay"])

    sc = CFG["scheduler"]
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode=sc["mode"], factor=sc["factor"],
                                                           patience=sc["patience"], min_lr=sc["min_lr"])
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    st = {"epoch": 0, "history": [], "best_val_loss": float("inf"), "best_epoch": 0, "bad": 0,
          "train_time": 0.0, "finished": False, "stop_reason": None}

    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"]); st = ck["state"]
        log(f"[fold {k}] resumed from epoch {st['epoch']} (finished={st['finished']})")

    hist = st["history"]
    es = CFG["early_stopping"]
    energy = Energy(f"fold{k}_train")
    if not st["finished"]:
        energy.start()
        for ep in range(st["epoch"] + 1, CFG["max_epochs"] + 1):
            t0 = time.time(); lr = optimizer.param_groups[0]["lr"]      # head lr (group 0)
            trl, tra, _ = run_epoch(model, tr_loader, criterion, optimizer, scaler, device, True, use_amp, K)
            val, vaa, vauc = run_epoch(model, va_loader, criterion, None, scaler, device, False, use_amp, K)
            scheduler.step(val)
            dt = time.time() - t0; st["train_time"] += dt
            hist.append({"epoch": ep, "train_loss": trl, "train_acc": tra, "val_loss": val, "val_acc": vaa,
                         "val_roc_auc": vauc, "lr": lr, "epoch_time_s": dt})
            improved = val < st["best_val_loss"] - es["min_delta"]
            if improved:
                st.update(best_val_loss=val, best_epoch=ep, bad=0)
                torch.save({"model": model.state_dict(), "epoch": ep, "val_loss": val, "val_acc": vaa,
                            "class_names": class_names, "config": CFG}, best_p)        # BEST MODEL SAVED HERE
            else:
                st["bad"] += 1
            st["epoch"] = ep
            log(f"[fold {k}] ep {ep:02d}/{CFG['max_epochs']} | train loss {trl:.4f} acc {tra:.4f} | "
                f"val loss {val:.4f} acc {vaa:.4f} auc {vauc:.4f} | lr {lr:.2e} | {dt:.1f}s"
                + (" *best*" if improved else f" (no improve {st['bad']}/{es['patience']})"))
            stop = st["bad"] >= es["patience"]
            if stop or ep == CFG["max_epochs"]:
                st["finished"] = True
                st["stop_reason"] = "early_stopping" if stop else "max_epochs"
            torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "state": st}, last_p)
            pd.DataFrame(hist).to_csv(os.path.join(fdir, "training_history.csv"), index=False)
            if st["finished"]:
                log(f"[fold {k}] stopped: {st['stop_reason']}"); break
        st["energy_train"] = energy.stop()

    h = pd.DataFrame(hist); h.to_csv(os.path.join(fdir, "training_history.csv"), index=False)
    plot_curves(h, os.path.join(fdir, "training_curves.png"), f"{CFG['model_name']} fold {k}")
    b = h[h["epoch"] == st["best_epoch"]].iloc[0]
    summary = {"fold": k, "epochs_run": int(len(h)), "stop_reason": st["stop_reason"],
               "best_epoch": int(st["best_epoch"]), "best_val_loss": float(b["val_loss"]),
               "val_acc_at_best_epoch": float(b["val_acc"]), "val_roc_auc_at_best_epoch": float(b["val_roc_auc"]),
               "best_val_acc_any_epoch": float(h["val_acc"].max()),
               "final_train_loss": float(h["train_loss"].iloc[-1]), "final_train_acc": float(h["train_acc"].iloc[-1]),
               "total_training_time_s": float(st["train_time"]),
               "avg_epoch_time_s": float(h["epoch_time_s"].mean()),
               "total_params": int(total_p), "trainable_params": int(train_p),
               "n_train": len(tr_df), "n_val": len(va_df), "energy_train": st.get("energy_train")}
    jdump(summary, os.path.join(fdir, "fold_summary.json"))
    return summary


# ---------------- TEST + EFFICIENCY ----------------
def load_best(k, run_dir, K, device):
    m = build_model(K)
    ck = torch.load(os.path.join(run_dir, f"fold_{k}", "best.pt"), map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model"]); return m.to(device).eval()


def save_test_outputs(out_dir, name, test_df, probs, class_names):
    os.makedirs(out_dir, exist_ok=True)
    y = test_df["y"].values; pred = probs.argmax(1)
    m = compute_metrics(y, probs, class_names)
    pdf = pd.DataFrame({"image_path": test_df["orig_path"].values, "resolved_path": test_df["path"].values,
                        "true_label": [class_names[i] for i in y], "pred_label": [class_names[i] for i in pred],
                        "confidence": probs.max(1), "correct": (pred == y)})
    for i, c in enumerate(class_names):
        pdf[f"prob_{c}"] = probs[:, i]
    pdf.to_csv(os.path.join(out_dir, "predictions.csv"), index=False)
    jdump(m, os.path.join(out_dir, "test_metrics.json"))
    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(m["classification_report"])
    pd.DataFrame(m["confusion_matrix"], index=class_names, columns=class_names).to_csv(
        os.path.join(out_dir, "confusion_matrix.csv"))
    plot_confusion(m["confusion_matrix"], class_names, os.path.join(out_dir, "confusion_matrix.png"), name)
    plot_roc(y, probs, class_names, os.path.join(out_dir, "roc_curve.png"), f"ROC - {name}")
    log(f"[{name}] TEST acc {m['accuracy']:.4f} | macroF1 {m['macro_f1']:.4f} | weightedF1 {m['weighted_f1']:.4f} "
        f"| ROC-AUC(macro OvR) {m['roc_auc_ovr_macro']:.4f}")
    return m


def platform_cpu():
    try:
        import platform, subprocess
        name = subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip()
        return f"{name or platform.processor()} | logical cores: {os.cpu_count()}"
    except Exception:
        return f"logical cores: {os.cpu_count()}"


def benchmark_efficiency(model, K, out_dir):
    """Params, FLOPs, model size, CPU latency/FPS, RAM, energy - batch size 1, CPU."""
    b = CFG["bench"]; proc = psutil.Process(os.getpid())
    m = copy.deepcopy(model).cpu().eval()
    x = torch.randn(1, 3, CFG["img_size"], CFG["img_size"])
    res = {"cpu": platform_cpu(), "torch_threads": torch.get_num_threads(),
           "input_shape": [1, 3, CFG["img_size"], CFG["img_size"]]}
    res["total_params"], res["trainable_params"] = count_params(m)
    flops = None
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad():
            m(x)
        flops = int(fc.get_total_flops())
    except Exception as e:
        log(f"  FlopCounterMode failed ({e}); trying thop")
        try:
            from thop import profile
            flops = int(profile(m, inputs=(x,), verbose=False)[0] * 2)
        except Exception:
            pass
    res["flops"] = flops; res["gflops"] = flops / 1e9 if flops else None
    res["macs"] = flops // 2 if flops else None
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as tf:
        torch.save(m.state_dict(), tf.name); res["model_size_mb"] = os.path.getsize(tf.name) / 1024 ** 2
    os.unlink(tf.name)

    peak = {"v": proc.memory_info().rss}; stop = threading.Event()

    def sampler():
        while not stop.is_set():
            peak["v"] = max(peak["v"], proc.memory_info().rss); time.sleep(0.01)
    base = proc.memory_info().rss; th = threading.Thread(target=sampler, daemon=True); th.start()
    with torch.no_grad():
        for _ in range(b["warmup"]):
            m(x)
        ts = []
        for _ in range(b["runs"]):
            t0 = time.perf_counter(); m(x); ts.append((time.perf_counter() - t0) * 1000)
    stop.set(); th.join()
    ts = np.array(ts)
    res.update({"cpu_latency_ms_mean": float(ts.mean()), "cpu_latency_ms_std": float(ts.std()),
                "cpu_latency_ms_median": float(np.median(ts)), "cpu_latency_ms_p95": float(np.percentile(ts, 95)),
                "fps": float(1000.0 / ts.mean()), "warmup_runs": b["warmup"], "timed_runs": b["runs"],
                "ram_baseline_mb": base / 1024 ** 2, "ram_peak_mb": peak["v"] / 1024 ** 2,
                "ram_increase_mb": (peak["v"] - base) / 1024 ** 2})
    res["energy_per_inference_J_estimated_TDP"] = float(ts.mean() / 1000 * b["assumed_cpu_tdp_watts"])
    res["assumed_cpu_tdp_watts"] = b["assumed_cpu_tdp_watts"]
    en = Energy("cpu_inference"); en.start(); n_inf = 0; t_end = time.time() + b["energy_seconds"]
    with torch.no_grad():
        while time.time() < t_end:
            m(x); n_inf += 1
    e = en.stop(); res["codecarbon"] = {**e, "inferences_in_energy_run": n_inf, "duration_s": b["energy_seconds"]}
    if e["energy_kwh"]:
        res["energy_per_inference_J_codecarbon"] = e["energy_kwh"] * 3.6e6 / n_inf
    res["note"] = ("Colab CPU numbers do not represent a target edge device. CodeCarbon on Colab usually "
                   "ESTIMATES power from CPU model/TDP (no hardware power sensor); treat energy as an estimate.")
    jdump(res, os.path.join(out_dir, "efficiency_report.json"))
    log(f"[efficiency] params {res['total_params']:,} | GFLOPs {res['gflops']} | size {res['model_size_mb']:.1f} MB | "
        f"CPU {res['cpu_latency_ms_mean']:.1f} ms | {res['fps']:.1f} FPS | RAM peak {res['ram_peak_mb']:.0f} MB")
    return res


# ---------------- MAIN ----------------
def main():
    global LOG_FILE
    _INDEX.clear(); _IMG_CACHE.clear()
    CFG.update(CFG["model_overrides"].get(CFG["model_name"], {}))
    if CFG["model_source"] == "timm":
        import timm; print("timm version:", timm.__version__)
    if not os.path.exists(CFG["data_root"]):
        from google.colab import drive
        drive.mount("/content/drive")
    run_dir = CFG["resume_run_dir"] or os.path.join(CFG["output_root"],
                                                    f"{CFG['model_name']}_{datetime.now():%Y%m%d_%H%M%S}")
    os.makedirs(run_dir, exist_ok=True); LOG_FILE = os.path.join(run_dir, "run_log.txt")
    log(f"Run directory: {run_dir}")
    log(f"Model: {CFG['model_name']} ({CFG['model_source']}) | img_size {CFG['img_size']} | batch {CFG['batch_size']} | "
        f"lr {CFG['lr']} | epochs {CFG['max_epochs']} | dropout {CFG['dropout']} | seed {CFG['seed']} | "
        f"unfreeze_last_blocks {CFG['unfreeze_last_blocks']}")

    folds, test_df, class_names = load_and_verify(run_dir)
    if VERIFY_ONLY:
        log("VERIFY_ONLY=True: stopping after verification."); return
    K = len(class_names); set_seed(CFG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    env = {"device": str(device), "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
           "torch": torch.__version__, "python": sys.version, "class_names": class_names,
           "run_started": datetime.now().isoformat()}
    jdump({"config": CFG, "environment": env}, os.path.join(run_dir, "config.json"))
    log(f"Device: {device} ({env['gpu']}) | torch {torch.__version__}")
    if device.type == "cpu":
        log("WARNING: no GPU detected - training will be very slow. Runtime > Change runtime type > GPU.")

    # ---- train + validate (3 folds) ----
    log("=" * 80); log("STEP 2: training"); log("=" * 80)
    fold_summ = {}
    for k, d in folds.items():
        fold_summ[k] = train_fold(k, d["train"], d["val"], class_names, run_dir, device)
        s = fold_summ[k]
        log(f"[fold {k}] best epoch {s['best_epoch']} | best val loss {s['best_val_loss']:.4f} | "
            f"val acc {s['val_acc_at_best_epoch']:.4f} | time {s['total_training_time_s']/60:.1f} min")

    # ---- test on Common_test_20 ----
    log("=" * 80); log("STEP 3: testing on common test set"); log("=" * 80)
    _, eval_tf = make_transforms()
    test_ds = ImgDS(test_df, eval_tf)
    tdir = os.path.join(run_dir, "test_results"); sdir = os.path.join(run_dir, "summary")
    os.makedirs(sdir, exist_ok=True)
    all_probs, rows = [], []
    for k in folds:
        model = load_best(k, run_dir, K, device)
        probs = predict(model, test_ds, device); all_probs.append(probs)
        met = save_test_outputs(os.path.join(tdir, f"fold_{k}"), f"{CFG['model_name']} fold {k}",
                                test_df, probs, class_names)
        rows.append({"model": f"fold_{k}", **{x: met[x] for x in (
            "accuracy", "precision_macro", "recall_macro", "macro_f1", "weighted_f1",
            "roc_auc_ovr_macro", "roc_auc_ovr_weighted")}})
    ens = np.mean(all_probs, axis=0)
    met = save_test_outputs(os.path.join(tdir, "ensemble_softvote"), f"{CFG['model_name']} 3-fold soft-vote ensemble",
                            test_df, ens, class_names)
    ens_row = {"model": "ensemble_softvote", **{x: met[x] for x in rows[0] if x != "model"}}
    df = pd.DataFrame(rows)
    mean_row = {"model": "mean_of_folds", **df.drop(columns="model").mean().to_dict()}
    std_row = {"model": "std_of_folds", **df.drop(columns="model").std(ddof=1).to_dict()}
    final = pd.concat([df, pd.DataFrame([mean_row, std_row, ens_row])], ignore_index=True)
    final.to_csv(os.path.join(sdir, "test_summary_all_folds.csv"), index=False)
    pd.DataFrame(fold_summ).T.to_csv(os.path.join(sdir, "training_summary_all_folds.csv"), index=False)
    log("\n" + final.round(4).to_string(index=False))

    # ---- efficiency ----
    log("=" * 80); log("STEP 4: efficiency benchmark (fold-1 best model, CPU, batch=1)"); log("=" * 80)
    eff = benchmark_efficiency(load_best(min(folds), run_dir, K, device), K, sdir)
    train_energy = {f"fold_{k}": fold_summ[k].get("energy_train") for k in fold_summ}
    jdump({"efficiency_single_model": eff, "training_energy_per_fold": train_energy,
           "training_time_s_per_fold": {f"fold_{k}": fold_summ[k]["total_training_time_s"] for k in fold_summ},
           "ensemble_note": "Ensemble of 3 models ~= 3x params/FLOPs/latency/size of a single model."},
          os.path.join(sdir, "full_summary.json"))
    log(f"\nDONE. Everything saved in: {run_dir}")


main()

timm version: 1.0.30
Run directory: /content/drive/MyDrive/Colab Notebooks/Thesis notebook/efficientnet_b0_20261016_185143
Model: efficientnet_b0 (timm) | img_size 224 | batch 32 | lr 0.0001 | epochs 30 | dropout 0.3 | seed 42 | unfreeze_last_blocks 1
STEP 1: verifying dataset + CSV manifests
Images on disk under data_root: 865  | folders containing images: ['Aphids_Disease', 'Blotch', 'Healthy_Leaf', 'Leaf_Spot']
Folds found: [1, 2, 3]
  fold 1: train=Fold_1_Train.csv  val=Fold_1_Validation.csv
  fold 2: train=Fold_2_Train.csv  val=Fold_2_Validation.csv
  fold 3: train=Fold_3_Train.csv  val=Fold_3_Validation.csv
  Test set: using manifest /content/drive/MyDrive/Dataset Files/Turmeric Dataset/Common_test_20/Common_Test_20.csv

Classes (4): ['Aphids_Disease', 'Blotch', 'Healthy_Leaf', 'Leaf_Spot']

TEST  n=175  per-class=[45, 48, 43, 39]
FOLD 1: train=460 [117, 127, 113, 103] | val=230 [59, 63, 57, 51]
        overlaps -> train/val=0 train/test=0 val/test=0  dups(train/val)=0/0
FOLD 2: 

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

  [freeze] trainable 1,134,516/4,012,672 params
  [freeze] trainable layers: ['blocks.6.0.bn1', 'blocks.6.0.bn2', 'blocks.6.0.bn3', 'blocks.6.0.conv_dw', 'blocks.6.0.conv_pw', 'blocks.6.0.conv_pwl', 'blocks.6.0.se.conv_expand', 'blocks.6.0.se.conv_reduce', 'bn2', 'classifier', 'conv_head']
[fold 1] model parameters: total 4,012,672 | trainable 1,134,516
  (codecarbon unavailable: No module named 'codecarbon') - energy will be estimated only


/tmp/ipykernel_1800/3966927049.py:613: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


...

  [freeze] trainable 1,134,516/4,012,672 params
  [freeze] trainable layers: ['blocks.6.0.bn1', 'blocks.6.0.bn2', 'blocks.6.0.bn3', 'blocks.6.0.conv_dw', 'blocks.6.0.conv_pw', 'blocks.6.0.conv_pwl', 'blocks.6.0.se.conv_expand', 'blocks.6.0.se.conv_reduce', 'bn2', 'classifier', 'conv_head']
[fold 2] model parameters: total 4,012,672 | trainable 1,134,516
  (codecarbon unavailable: No module named 'codecarbon') - energy will be estimated only


/tmp/ipykernel_1800/3966927049.py:613: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


[fold 2] ep 01/30 | train loss 3.6442 acc 0.3087 | val loss 4.4381 acc 0.2609 auc 0.5298 | lr 1.00e-04 | 1.6s *best*
[fold 2] ep 02/30 | train loss 3.0779 acc 0.3565 | val loss 3.4817 acc 0.3087 auc 0.5920 | lr 1.00e-04 | 1.6s *best*
[fold 2] ep 03/30 | train loss 2.2855 acc 0.4370 | val loss 2.8010 acc 0.3478 auc 0.6555 | lr 1.00e-04 | 1.6s *best*
[fold 2] ep 04/30 | train loss 2.0465 acc 0.5087 | val loss 2.4041 acc 0.3913 auc 0.7107 | lr 1.00e-04 | 1.7s *best*
[fold 2] ep 05/30 | train loss 1.8418 acc 0.5087 | val loss 2.1306 acc 0.4348 auc 0.7472 | lr 1.00e-04 | 1.6s *best*
[fold 2] ep 06/30 | train loss 1.3659 acc 0.6413 | val loss 1.8733 acc 0.4957 auc 0.7844 | lr 1.00e-04 | 1.7s *best*
[fold 2] ep 07/30 | train loss 1.3987 acc 0.6174 | val loss 1.6735 acc 0.5348 auc 0.8159 | lr 1.00e-04 | 1.6s *best*
[fold 2] ep 08/30 | train loss 1.2733 acc 0.6739 | val loss 1.4993 acc 0.5696 auc 0.8391 | lr 1.00e-04 | 1.6s *best*
[fold 2] ep 09/30 | train loss 0.9663 acc 0.7130 | val loss 1.33

/tmp/ipykernel_1800/3966927049.py:613: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


[fold 3] ep 01/30 | train loss 3.7253 acc 0.2891 | val loss 3.9921 acc 0.2783 auc 0.5750 | lr 1.00e-04 | 1.6s *best*
[fold 3] ep 02/30 | train loss 3.1169 acc 0.3761 | val loss 3.0132 acc 0.3391 auc 0.6339 | lr 1.00e-04 | 1.7s *best*
[fold 3] ep 03/30 | train loss 2.4893 acc 0.4261 | val loss 2.3992 acc 0.4217 auc 0.7031 | lr 1.00e-04 | 1.6s *best*
[fold 3] ep 04/30 | train loss 2.0625 acc 0.4848 | val loss 2.1203 acc 0.4565 auc 0.7524 | lr 1.00e-04 | 1.8s *best*
[fold 3] ep 05/30 | train loss 1.7475 acc 0.5413 | val loss 1.7886 acc 0.5261 auc 0.7956 | lr 1.00e-04 | 1.6s *best*
[fold 3] ep 06/30 | train loss 1.4309 acc 0.6043 | val loss 1.5454 acc 0.5696 auc 0.8299 | lr 1.00e-04 | 1.6s *best*
[fold 3] ep 07/30 | train loss 1.3298 acc 0.6457 | val loss 1.3227 acc 0.6348 auc 0.8572 | lr 1.00e-04 | 1.7s *best*
[fold 3] ep 08/30 | train loss 1.1636 acc 0.6587 | val loss 1.2175 acc 0.6565 auc 0.8752 | lr 1.00e-04 | 1.6s *best*
[fold 3] ep 09/30 | train loss 1.0638 acc 0.6891 | val loss 1.08